# 25 SFT evaluation and method selection (CPU runtime)

For every trained model with predictions: validation (held-out train databases: correctness and step counts), dev (EX) and mini-dev (EX and quick timing). Then the selection rule, fixed before any training: among the methods whose validation EX is not significantly below the best (paired McNemar test), the one with the most answers at least 2x cheaper than gold wins. Dev and mini-dev are reported, never used to choose.

**Runtime > Change runtime type > CPU**: timing runs on the same machine type as all earlier timing. Rerun this notebook after the winner's second epoch (`26_sft_epoch2`): it adds that model and compares epoch 1 with epoch 2 on validation.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])
E1 = [f"{EXPORT}_{a}_e1" for a in ("gold", "best", "weighted", "tagged")]
E2 = sorted(d for d in os.listdir(R) if d.startswith(EXPORT) and d.endswith("_e2"))
RUNS = [r for r in E1 + E2 if os.path.exists(f"{R}/{r}/predictions_val.jsonl")]
print("validation databases:", VAL_DBS)
print("models with predictions:", RUNS)

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score every model (resumable; finished parts are reused)

In [ ]:
for run in RUNS:
    arm = json.load(open(f"{R}/{run}/adapter_meta.json"))["arm"]
    !python -m effsql.train eval-val --run {run}
    !effsql evaluate --pred {R}/{run}/predictions_dev.jsonl --split dev --timing none --run {run}_dev --model {MODEL} --method sft-{arm}
    !effsql evaluate --pred {R}/{run}/predictions_minidev.jsonl --split minidev --timing quick --run {run}_minidev --model {MODEL} --method sft-{arm}

### 3. Method selection on validation (needs all four epoch-1 models)

In [ ]:
if all(r in RUNS for r in E1):
    !python -m effsql.train select --runs {",".join(E1)} --out {EXPORT}_selection
else:
    print("still missing:", [r for r in E1 if r not in RUNS])

### 4. Epoch 2 against epoch 1 (after `26_sft_epoch2`): the same rule decides which to keep

In [ ]:
if E2:
    W = json.load(open(f"{R}/{EXPORT}_selection/selection.json"))["winner"]
    !python -m effsql.train select --runs {W},{E2[0]} --out {EXPORT}_epoch_choice
else:
    print("no second-epoch model yet")

### 5. All results so far (dev and mini-dev next to the zero-shot baseline)

In [ ]:
import pandas as pd
df = pd.read_csv(f"{DRIVE_ROOT}/effsql/results/results.csv")
df = df[df["run"].str.startswith((EXPORT, "pool_v1_"))].drop_duplicates("run", keep="last")
cols = ["run", "split", "n", "ex", "rel_cost_median", "rel_cost_geomean", "eff_acc@1.5", "quick_ratio_median", "cpu_model"]
print(df[cols].to_string(index=False))